# 📘 Text Generation with Vanilla RNN, LSTM, and GRU
## A complete, comparative deep-learning study

This notebook designs and implements three sequence models — **Vanilla RNN**, **LSTM**, and **GRU** — 
trained on the *same* corpus for **next-word prediction**, then compares them on:

- training **loss** and **accuracy**
- **perplexity** (how confidently/correctly the model predicts the next word)
- **parameter count** (model complexity)
- **training time** (speed)
- **generated text quality** (coherence & grammar)

🎯 **Goal:** Show *with evidence* why gated architectures (LSTM/GRU) handle longer-range 
dependencies better than a vanilla RNN, and where GRU's simplicity pays off.

# 🧠 Problem Statement
Design and implement a DL model capable of learning the underlying structure, grammar, and 
contextual dependencies of a given text corpus to generate coherent and meaningful text 
sequences using **Vanilla RNN**, **LSTM**, and **GRU**.

Then compare: training loss, generated text quality, memory handling, and long-term dependency learning.

# ⚙️ Setup & Reproducibility
We fix random seeds so results are repeatable across runs — important when *comparing* models.

In [ ]:
import os, time, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import matplotlib.pyplot as plt

# --- reproducibility ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)

# 📥 Text Corpus
Unlike a 6-line toy corpus, we use a **larger, structured passage**. A bigger vocabulary and 
longer sentences are what actually let us observe differences in **memory handling** and 
**long-term dependencies** between the three models.

> 💡 You can replace `corpus` below with any plain text (a story, an article, lyrics you have rights to, 
PDF-extracted text, etc.). More data = better generation.

In [ ]:
corpus = '''
deep learning is transforming the field of artificial intelligence
neural networks learn patterns directly from raw data
recurrent neural networks are designed for sequential information
they process one element at a time while keeping a hidden state
the hidden state acts as a form of memory across the sequence
a vanilla recurrent network struggles to remember distant context
this happens because gradients vanish over many time steps
long short term memory networks were created to fix this problem
an lstm uses an input gate a forget gate and an output gate
these gates decide what to keep and what to discard from memory
as a result an lstm can capture long range dependencies in text
the gated recurrent unit is a simpler alternative to the lstm
a gru combines the gates into a reset gate and an update gate
with fewer parameters a gru trains faster than an lstm
in many tasks a gru reaches similar accuracy to an lstm
language models predict the next word given the previous words
to generate text the model feeds its own prediction back as input
good generation requires both correct grammar and meaningful context
training on more data usually improves the quality of generated text
comparing these architectures helps us understand sequence modeling
deep learning models can generate coherent and meaningful sentences
understanding memory and context is the key to better text generation
'''

print(corpus.strip())

# 🔤 Tokenization & Sequence Creation
We convert text to integer tokens and build **n-gram sequences** for next-word prediction. 
Each sequence's last token is the target `y`; everything before it is the input `X`.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print('Vocabulary size:', total_words)

input_sequences = []
for line in corpus.split('\n'):
    line = line.strip()
    if not line:
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print('Number of training sequences:', X.shape[0])
print('Input length (timesteps):', X.shape[1])
print('X shape:', X.shape, '| y shape:', y.shape)

# 🏗️ Shared Configuration & Model Builder
To make the comparison **fair**, all three models share the same embedding size, number of 
hidden units, optimizer, loss, and number of epochs. Only the recurrent layer changes.

We also write a small helper that **times training** so we can compare speed later.

In [ ]:
EMBED_DIM   = 64
HIDDEN_UNITS = 128
EPOCHS      = 200

def build_model(recurrent_layer):
    """Build an identical architecture, swapping only the recurrent layer."""
    model = Sequential([
        Embedding(total_words, EMBED_DIM),   # input_length removed (deprecated in TF 2.20)
        recurrent_layer,
        Dense(total_words, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model

def train_timed(model, name):
    """Train the model and return (history, seconds_taken)."""
    start = time.time()
    history = model.fit(X, y, epochs=EPOCHS, verbose=0)
    elapsed = time.time() - start
    print(f'{name} training completed in {elapsed:.1f}s')
    return history, elapsed

# 🧠 Model 1: Vanilla RNN
The baseline. A `SimpleRNN` keeps a single hidden state and tends to **forget distant context** 
because gradients vanish over long sequences.

In [ ]:
rnn_model = build_model(SimpleRNN(HIDDEN_UNITS))
rnn_history, rnn_time = train_timed(rnn_model, 'Vanilla RNN')

# 🔒 Model 2: LSTM
LSTM adds **input, forget, and output gates** plus a cell state, letting it preserve information 
over long spans — better long-term dependency handling.

In [ ]:
lstm_model = build_model(LSTM(HIDDEN_UNITS))
lstm_history, lstm_time = train_timed(lstm_model, 'LSTM')

# ⚡ Model 3: GRU
GRU merges the gating into a **reset gate** and an **update gate**. Fewer parameters than LSTM 
→ usually **faster training** with comparable accuracy.

In [ ]:
gru_model = build_model(GRU(HIDDEN_UNITS))
gru_history, gru_time = train_timed(gru_model, 'GRU')

## 📉 Training Loss Comparison
Lower loss means the model is more confident and correct about the next word during training.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(rnn_history.history['loss'],  label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'],  label='GRU')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 📈 Training Accuracy Comparison
How often the model's top prediction matches the true next word.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(rnn_history.history['accuracy'],  label='RNN')
plt.plot(lstm_history.history['accuracy'], label='LSTM')
plt.plot(gru_history.history['accuracy'],  label='GRU')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('Training Accuracy Comparison')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 📊 Quantitative Comparison Table
We summarise each model with four metrics:

- **Params** — model complexity (more isn't always better).
- **Train time (s)** — speed.
- **Final loss / accuracy** — fit on the training data.
- **Perplexity** = `exp(loss)` — a standard language-model metric; **lower is better** 
  (≈ how many words the model is 'choosing between' on average).

In [ ]:
import math

def summarize(model, history, seconds):
    final_loss = history.history['loss'][-1]
    final_acc  = history.history['accuracy'][-1]
    return {
        'params': model.count_params(),
        'time_s': round(seconds, 1),
        'loss': round(final_loss, 4),
        'accuracy': round(final_acc, 4),
        'perplexity': round(math.exp(final_loss), 2),
    }

results = {
    'RNN':  summarize(rnn_model,  rnn_history,  rnn_time),
    'LSTM': summarize(lstm_model, lstm_history, lstm_time),
    'GRU':  summarize(gru_model,  gru_history,  gru_time),
}

# pretty-print as a table (no extra libraries needed)
cols = ['params', 'time_s', 'loss', 'accuracy', 'perplexity']
header = 'Model'.ljust(8) + ''.join(c.ljust(13) for c in cols)
print(header)
print('-' * len(header))
for name, r in results.items():
    row = name.ljust(8) + ''.join(str(r[c]).ljust(13) for c in cols)
    print(row)

# ✍️ Text Generation (with temperature sampling)
Pure `argmax` always picks the single most likely word, which makes output repetitive and 
deterministic. Adding a **temperature** lets us control randomness:

- `temperature → 0` : greedy, safe, repetitive (like argmax)
- `temperature = 1` : sample from the model's true distribution
- `temperature > 1` : more diverse but more error-prone

We build an `index → word` lookup once for efficiency.

In [ ]:
index_to_word = {idx: word for word, idx in tokenizer.word_index.items()}

def sample_with_temperature(probs, temperature):
    if temperature <= 0:
        return int(np.argmax(probs))
    probs = np.asarray(probs).astype('float64')
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs)
    probs = probs / np.sum(probs)
    return int(np.random.choice(len(probs), p=probs))

def generate_text(model, seed_text, next_words=8, temperature=0.0):
    result = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')
        probs = model.predict(token_list, verbose=0)[0]
        next_index = sample_with_temperature(probs, temperature)
        next_word = index_to_word.get(next_index, '')
        if not next_word:
            break
        result += ' ' + next_word
    return result

## 🧪 Generate Text Samples
Same seed for all three models so the comparison is fair. We show a **greedy** pass 
(temperature 0) for a clean grammar check, plus a **creative** pass (temperature 0.7).

In [ ]:
seed = 'deep learning'

print('=== Greedy (temperature = 0.0) ===')
print('RNN :', generate_text(rnn_model,  seed, 8, temperature=0.0))
print('LSTM:', generate_text(lstm_model, seed, 8, temperature=0.0))
print('GRU :', generate_text(gru_model,  seed, 8, temperature=0.0))

print('\n=== Creative (temperature = 0.7) ===')
np.random.seed(SEED)  # keep sampling reproducible
print('RNN :', generate_text(rnn_model,  seed, 8, temperature=0.7))
print('LSTM:', generate_text(lstm_model, seed, 8, temperature=0.7))
print('GRU :', generate_text(gru_model,  seed, 8, temperature=0.7))

## 🔎 Generated Text Quality — what to look for
When you run the cells above, evaluate the output against these questions (this is the 
qualitative half of the comparison the problem statement asks for):

1. **Grammar** — Does the sentence read like valid English (article + noun, subject + verb)?
2. **Local coherence** — Do neighbouring words belong together (e.g. *forget gate*, *hidden state*)?
3. **Long-range context** — Does the model stay on-topic for the whole generated span, or drift/repeat?
4. **Repetition** — Vanilla RNN often loops the same few words; gated models usually drift less.

Typical observations on a corpus this size:
- **RNN** produces locally plausible pairs but loses the thread and repeats sooner.
- **LSTM** holds the topic longest and forms the most complete clauses.
- **GRU** is close to LSTM in quality while training faster (check the table above).

# 📚 Student Learning Tasks
### ✅ Beginner
1. Replace `corpus` with your own paragraphs and re-run.
2. Change `EMBED_DIM`, `HIDDEN_UNITS`, or `EPOCHS` and watch the table change.
3. Generate more words and try temperatures 0.2, 0.7, 1.2.

### ✅ Intermediate
4. Add a validation split (`model.fit(..., validation_split=0.2)`) and plot val-loss to spot overfitting.
5. Stack two recurrent layers (`return_sequences=True` on the first).
6. Add `Dropout` and compare perplexity.

### ✅ Advanced
7. Train on a much larger text file and compare how the gap between RNN and LSTM/GRU widens.
8. Add early stopping and compare training time fairly at matched validation loss.

# ✅ Conclusion
This notebook implements **all three architectures** on a shared corpus and compares them 
**quantitatively** (loss, accuracy, perplexity, parameters, training time) and 
**qualitatively** (generated text grammar and coherence).

- **Vanilla RNN** — learns short local patterns but loses distant context (vanishing gradients), 
  so it repeats and drifts sooner.
- **LSTM** — gates + cell state preserve long-range dependencies, giving the most coherent text, 
  at the cost of the most parameters and slowest training.
- **GRU** — fewer gates and parameters than LSTM, **trains faster**, and reaches comparable 
  accuracy/perplexity — often the best speed/quality trade-off.

Together these results **demonstrate** (not just assert) why gated architectures outperform a 
vanilla RNN on sequence modeling and text generation.

> 📌 Note: With a small corpus all models can fit the data well, so differences are clearest in 
**generation quality**, **perplexity**, and **training time**. The gap grows as the corpus gets larger.